# Gamil RAG project

### Step 1: Import library

In [ ]:
import os
import ollama
from openai import OpenAI
import json
from dotenv import load_dotenv
from huggingface_hub import login
import gradio as gr
from gmail.gmail_function import login_gmail, get_emails_list
from tqdm import tqdm
from IPython.display import display, Markdown
from pydantic import BaseModel, Field
from chromadb import PersistentClient

### Step 2: setting variables and environment

In [ ]:
load_dotenv(override = True)
ollama_api_key = os.getenv("OLLAMA_API_KEY")
hf_token = os.getenv("HF_TOKEN")
credential_path = r'.credentials/credentials.json' # create credentials from Gmail API
token_path = r'.credentials/token.json' # Token will be created the first time of login to be used for subsequent login
login(hf_token, add_to_git_credential=True)

# # setup ollama
llm_client = OpenAI(base_url = "http://localhost:11434/v1/", api_key = ollama_api_key)
llm_model = "gemma3:1b"

# database and embeddings setting
DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
# setup lms
# llm_client = OpenAI(base_url = "http://localhost:1234/v1/", api_key = "lms")
# llm_model = "google/gemma-3-4b"

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### Step 3: Check if gmail credentials exist

In [55]:
if not os.path.exists(credential_path):
    print(f"Error: Credentials file '{credential_path}' does not exist.")
else:
    print("Credential file exists!")

Credential file exists!


### Step 4: Login gmail with credentials or token

In [4]:
service = login_gmail(credential_path, token_path)

### Step 5: Retrieve Gmail for the past 7 days

In [80]:
emails = get_emails_list(service, days=1)

In [ ]:
# print(len(emails))

35


### Step 6: Create a summary for every gmail

In [81]:
class Result(BaseModel):
    page_content: str = Field(description = "email sender, date received and summary")
    metadata: dict

class Email(BaseModel):
    title: str = Field(description = "Email title")
    sender: str = Field(description = "email sender details")
    date_received: str = Field(description = "Date of email received")
    body: str = Field(description = "Email body")
    priority: int = Field(description = "priority of the email")
    summary: str = Field(description = "Email summary based on title and body")

    def as_result(self):
        metadata = {"sender": self.sender, "date_received": self.date_received, "priority" : self.priority}
        return Result(page_content = self.sender + " send an email on " + self.date_received + ".\nEmail summary: " + self.summary, metadata = metadata)

class Emails(BaseModel):
    emails: list[Email]

class Email_llm_output(BaseModel):
    priority: int = Field(description = "priority of the email")
    summary: str = Field(description = "Email summary based on title and body")

In [82]:
system_prompt = f"""
You will be given the title and body of an email, some emails might not have any plain text for its body.
Based on the information, respond in json format only that contains priority from 1 to 5 ranking and a summary with the structure below.
For example,
{{
  "priority": integer between 1 to 5
  "summary": string that contains summary of email title and body
}}
"""

def preprocess_emails(emails):
  
  for email in tqdm(emails):
    title = email["title"]
    body = email["body"]
    user_prompt = f""" 
    Please help to provide priority ranking and summary of the email based on title and/or body. 
    Keep the summary to one or several sentences that are clear, concise and straight to the point. 

    Here is the title of the email:
    {title}

    Here is the body of the email:
    {body if body else "This email body does not contain any text. Please summarize based on the title only."}
    """
    messages = [{"role":"system", "content": system_prompt},
            {"role": "user", "content": user_prompt}]

    # response = llm_client.chat.completions.create(model = llm_model, messages = messages, max_tokens = 2000)
    response = ollama.chat(model = llm_model, messages = messages, format=Email_llm_output.model_json_schema())
    raw_content = response["message"]["content"]
    data = Email_llm_output.model_validate_json(raw_content)
    email["priority"] = data.priority
    email["summary"] = data.summary

  emails = Emails.model_validate({"emails":emails})
  return emails

In [ ]:
# emails = emails[:3]

In [83]:
emails = preprocess_emails(emails)



100%|██████████| 8/8 [00:21<00:00,  2.66s/it]


In [84]:
print(emails.emails[0].as_result())

page_content='PBe Online Banking <pbe@publicbank.com.my> send an email on 2026-03-20 20:48:06.\nEmail summary: This email is a warning from Public Bank Group regarding potential fraudulent activities during the Hari Raya celebrations. The sender urges recipients to be vigilant and cautious due to the prevalence of scams. The email includes a screenshot of a website providing scam prevention tips.' metadata={'sender': 'PBe Online Banking <pbe@publicbank.com.my>', 'date_received': '2026-03-20 20:48:06', 'priority': 4}


### Create database based on the Email list

In [ ]:
def create_embeddings(emails):
    chroma = PersistentClient(path = DB_NAME)
    

### Step 8: Create a chatbot for user to ask further questions

In [15]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] +\
                [{"role":h["role"], "content": h["content"]} for h in history]+\
                    [{"role": "user", "content": message}]
    response = llm_client.chat.completions.create(model = llm_model, messages = messages, max_tokens = 2000)
    return response.choices[0].message.content

In [16]:
with gr.Blocks() as ui:
    with gr.Row():
        gr.ChatInterface(fn=chat)
        
ui.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
